# Análise FTIR de Vinhos com Processamento Digital de Sinais

Este notebook reproduz a pipeline principal do projeto:

**dados FTIR brutos -> média das triplicatas -> filtros de suavização -> passa-bandas químicos -> extração de métricas -> gráficos finais**

A ideia é manter as etapas separadas para facilitar a apresentação e a reprodução dos resultados.

## 1. Configuração do ambiente

A célula abaixo localiza a raiz do projeto, adiciona `src/` ao `sys.path` e cria os diretórios necessários para salvar os resultados.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Funciona tanto executando o notebook pela raiz do projeto quanto pela pasta notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
IMAGES_DIR = PROJECT_ROOT / "images"

for path in [SRC_DIR, DATA_DIR, RESULTS_DIR, IMAGES_DIR]:
    path.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import utils

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 160
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = ":"
plt.rcParams["grid.alpha"] = 0.6

## 2. Carregamento dos dados brutos

O arquivo `Wine_FTIR_Triplicate_Spectra.csv` contém 37 vinhos medidos em triplicata. Cada coluna é uma réplica e cada linha é um ponto de número de onda.

In [ ]:
file_path = DATA_DIR / "Wine_FTIR_Triplicate_Spectra.csv"
df = utils.load_data(file_path)

wine_tags = []
for col in df.columns:
    wine_name = col.rsplit("_", 1)[0]
    if wine_name not in wine_tags:
        wine_tags.append(wine_name)

print(f"Formato dos dados brutos: {df.shape[0]} pontos x {df.shape[1]} réplicas")
print(f"Vinhos únicos: {len(wine_tags)}")
print(f"Faixa de wavenumbers: {df.index.min():.3f} a {df.index.max():.3f} cm^-1")
print(f"Cabernet: {sum('Cab' in w for w in wine_tags)} | Shiraz: {sum('Syr' in w for w in wine_tags)}")

df.head()

## 3. Visualização inicial: dados brutos como pontos

Antes de qualquer filtro, é útil mostrar que o espectro chega como uma sequência discreta de amostras de absorbância.

In [ ]:
raw_columns = ["Wine_01_Cab_Rep1", "Wine_01_Cab_Rep2", "Wine_01_Cab_Rep3"]
raw_output_names = [
    "dados_iniciais_wine_01_rep1.png",
    "dados_iniciais_wine_01_rep2.png",
    "dados_iniciais_wine_01_rep3.png",
]

for column, output_name in zip(raw_columns, raw_output_names):
    fig, ax = utils.plot_raw_triplicate_scatter(df[[column]], show=False)
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    plt.show()

## 4. Média das triplicatas

A média das três réplicas reduz ruído aleatório e gera um espectro médio por vinho.

\[
y_{mean}[n] = \frac{x_1[n] + x_2[n] + x_3[n]}{3}
\]

In [ ]:
df_mean = utils.calc_mean(df)
print(f"Formato após média das triplicatas: {df_mean.shape[0]} pontos x {df_mean.shape[1]} vinhos")
df_mean.head()

In [ ]:
def plot_triplicate_mean(df_raw, df_mean, wine_column, output_name=None, xlim=None):
    fig, ax = plt.subplots(figsize=(12, 6))

    colors = ["tab:blue", "black", "tab:green"]
    for rep, color in zip(["Rep1", "Rep2", "Rep3"], colors):
        ax.plot(
            df_raw.index,
            df_raw[f"{wine_column}_{rep}"],
            label=f"Réplica {rep[-1]}",
            color=color,
            linewidth=1.8,
        )

    ax.plot(
        df_mean.index,
        df_mean[wine_column],
        label="Média das triplicatas",
        color="tab:red",
        linestyle="--",
        linewidth=2.2,
    )

    if xlim is not None:
        ax.set_xlim(*xlim)

    ax.set_title(f"Comparação de Dados Crus e Média das Triplicatas - {wine_column}")
    ax.set_xlabel("Wavenumbers ($cm^{-1}$)")
    ax.set_ylabel("Absorbância")
    ax.legend()
    fig.tight_layout()

    if output_name:
        fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")

    return fig, ax

plot_triplicate_mean(df, df_mean, "Wine_01_Cab", "wine_01_media_triplicata.png")
plt.show()

plot_triplicate_mean(df, df_mean, "Wine_15_Syr", "wine_15_syr_media_triplicatas.png")
plt.show()

plot_triplicate_mean(df, df_mean, "Wine_30_Cab", "wine_30_cab_media_triplicatas.png")
plt.show()

### Zooms da média das triplicatas

Os zooms evidenciam variações pequenas entre réplicas que ficam pouco visíveis no gráfico completo.

In [ ]:
plot_triplicate_mean(
    df,
    df_mean,
    "Wine_01_Cab",
    "wine_01_media_triplicatas_ZOOM_IN_1.png",
    xlim=(1010, 1110),
)
plt.show()

plot_triplicate_mean(
    df,
    df_mean,
    "Wine_01_Cab",
    "wine_01_media_triplicatas_ZOOM_IN_2.png",
    xlim=(1360, 1480),
)
plt.show()

## 5. Filtros de suavização

São comparados dois filtros passa-baixas discretos:

- **Savitzky-Golay**: ajuste polinomial local, preserva melhor picos;
- **Média móvel FIR**: convolução com janela retangular, simples e eficiente, mas pode achatar picos.

In [ ]:
def plot_filter_comparison(df_raw, df_mean, wine_column, output_name=None):
    sinal_medio = df_mean[wine_column]
    sinal_sg = utils.dsp_filter(sinal_medio, janela=11, ordem=2)
    sinal_ma = utils.dsp_filter_moving_average(sinal_medio, janela=7)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        df_raw.index,
        df_raw[f"{wine_column}_Rep1"],
        label="Réplica 1 (bruto)",
        color="tab:green",
        linewidth=1.8,
    )
    ax.plot(
        sinal_sg.index,
        sinal_sg.values,
        label="Filtro Savitzky-Golay",
        color="tab:blue",
        linewidth=2,
    )
    ax.plot(
        sinal_ma.index,
        sinal_ma.values,
        label="Média móvel via convolução FIR",
        color="tab:red",
        linestyle="--",
        linewidth=2,
    )

    ax.set_title(f"Comparação de Filtros SLIT-D - {wine_column}")
    ax.set_xlabel("Wavenumbers ($cm^{-1}$)")
    ax.set_ylabel("Absorbância")
    ax.legend()
    fig.tight_layout()

    if output_name:
        fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")

    return fig, ax

plot_filter_comparison(df, df_mean, "Wine_01_Cab", "wine_01.png")
plt.show()

plot_filter_comparison(df, df_mean, "Wine_15_Syr", "wine_15_syr.png")
plt.show()

plot_filter_comparison(df, df_mean, "Wine_30_Cab", "wine_30_cab.png")
plt.show()

## 6. Extração de métricas por bandas químicas

Para cada vinho e para cada filtro, são extraídos picos e áreas nas bandas de açúcares, ácidos, polifenóis, proteínas e ésteres.

In [ ]:
resultados_sg = []
resultados_ma = []

for wine_column in df_mean.columns:
    sinal_sg = utils.dsp_filter(df_mean[wine_column])
    sinal_ma = utils.dsp_filter_moving_average(df_mean[wine_column])

    metrics_sg = utils.extract_wine_metrics(sinal_sg)
    metrics_ma = utils.extract_wine_metrics(sinal_ma)

    metrics_sg["Vinho"] = wine_column
    metrics_ma["Vinho"] = wine_column

    resultados_sg.append(metrics_sg)
    resultados_ma.append(metrics_ma)

df_resultados_sg = pd.DataFrame(resultados_sg).set_index("Vinho")
df_resultados_ma = pd.DataFrame(resultados_ma).set_index("Vinho")

# Mantém o nome do vinho no CSV para facilitar reuso posterior.
df_resultados_sg.reset_index().to_csv(RESULTS_DIR / "savitzky_golay_corrigido.csv", index=False)
df_resultados_ma.reset_index().to_csv(RESULTS_DIR / "media_movel_corrigido.csv", index=False)

print("Métricas Savitzky-Golay:")
display(df_resultados_sg.head())

print("Métricas Média Móvel:")
display(df_resultados_ma.head())

## 7. Passa-bandas por regiões químicas

As bandas são representadas por máscaras no eixo de número de onda. Visualmente, isso equivale a aplicar janelas passa-banda sobre o espectro suavizado.

In [ ]:
BAND_FILES = {
    "Açúcares": ("ftir_acucares.csv", "tab:blue"),
    "Ácidos org.": ("ftir_acidos_organicos.csv", "tab:green"),
    "Polifenóis": ("ftir_polifenois.csv", "tab:red"),
    "Proteínas": ("ftir_proteinas.csv", "purple"),
    "Ésteres": ("ftir_esteres.csv", "orange"),
}

band_windows = {
    label: pd.read_csv(DATA_DIR / filename).set_index("Wavenumbers")
    for label, (filename, color) in BAND_FILES.items()
}

def plot_pass_band_single(wine_column="Wine_01_Cab", output_name="passa_bandas_degraus_unitarios.png"):
    sinal = utils.dsp_filter(df_mean[wine_column])

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(
        sinal.index,
        sinal.values,
        label=f"Sinal com filtro SG - {wine_column}",
        color="black",
        alpha=0.55,
        linewidth=1.5,
    )

    for label, window in band_windows.items():
        color = BAND_FILES[label][1]
        ax.plot(
            window.index,
            window["Absorbance"],
            label=f"Passa-banda: {label}",
            color=color,
            linewidth=2,
        )

    ax.set_title("Mapeamento Espectral: Filtros Passa-Banda via Degrau Unitário")
    ax.set_xlabel("Frequência espacial / Wavenumbers ($cm^{-1}$)")
    ax.set_ylabel("Absorbância / ganho da janela")
    ax.set_xlim(sinal.index.min(), sinal.index.max())
    ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    return fig, ax

plot_pass_band_single()
plt.show()

In [ ]:
def plot_pass_band_comparison(
    wine_a="Wine_01_Cab",
    wine_b="Wine_15_Syr",
    output_name="compara_passa_bandas_wine01_wine15.png",
):
    sinal_a = utils.dsp_filter(df_mean[wine_a])
    sinal_b = utils.dsp_filter(df_mean[wine_b])

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(sinal_a.index, sinal_a.values, label="Garrafa 1 Cabernet Sauvignon", color="black", alpha=0.55)
    ax.plot(sinal_b.index, sinal_b.values, label="Garrafa 15 Shiraz", color="black", alpha=0.55, linestyle="--")

    for label, window in band_windows.items():
        color = BAND_FILES[label][1]
        ax.plot(window.index, window["Absorbance"], label=f"Passa-banda: {label}", color=color, linewidth=2)

    ax.set_title("Comparação entre Vinhos e Regiões Químicas FTIR")
    ax.set_xlabel("Frequência espacial / Wavenumbers ($cm^{-1}$)")
    ax.set_ylabel("Absorbância / ganho da janela")
    ax.set_xlim(sinal_a.index.min(), sinal_a.index.max())
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    return fig, ax

plot_pass_band_comparison()
plt.show()

## 8. Assinatura espectral por classe

As áreas das bandas são agrupadas por variedade (`Cabernet` ou `Shiraz`) para comparar a assinatura média das classes.

In [ ]:
AREA_COLUMNS = [
    "area_acucar",
    "area_acidos",
    "area_polifenois",
    "area_proteinas",
    "area_aroma",
]
AREA_LABELS = ["Açúcares", "Ácidos Orgânicos", "Polifenóis", "Proteínas", "Aromas (Ésteres)"]

for df_metrics in [df_resultados_sg, df_resultados_ma]:
    df_metrics["Uva"] = ["Cabernet" if "Cab" in wine else "Shiraz" for wine in df_metrics.index]

df_sg_media = df_resultados_sg.groupby("Uva")[AREA_COLUMNS].mean()
df_ma_media = df_resultados_ma.groupby("Uva")[AREA_COLUMNS].mean()

display(df_sg_media)
display(df_ma_media)

In [ ]:
def plot_compound_bars(df_medias, title, output_name):
    colors = ["#800020", "#4B0082"]
    ax = df_medias.T.plot(kind="bar", figsize=(10, 6), color=colors, edgecolor="black")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel("Energia Integrada da Banda (Absorbância x cm$^{-1}$)")
    ax.set_xlabel("Macronutrientes e Aromas")
    ax.set_xticks(range(len(AREA_LABELS)))
    ax.set_xticklabels(AREA_LABELS, rotation=15, ha="right")
    ax.legend(title="Classe da Uva")
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    return fig, ax

plot_compound_bars(df_sg_media, "Assinatura Espectral Média: Cabernet vs Shiraz (SG)", "concentracao_compostos_sg.png")
plt.show()

plot_compound_bars(df_ma_media, "Assinatura Espectral Média: Cabernet vs Shiraz (Média Móvel)", "concentracao_compostos_ma.png")
plt.show()

## 9. Espaço de separação: polifenóis x aromas

Este gráfico avalia se duas métricas extraídas das bandas já são suficientes para separar visualmente Cabernet e Shiraz.

In [ ]:
def plot_feature_space(df_metrics, output_name="espaco_separacao_sg.png"):
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = {"Cabernet": "#800020", "Shiraz": "#4B0082"}

    for grape, group in df_metrics.groupby("Uva"):
        ax.scatter(
            group["area_polifenois"],
            group["area_aroma"],
            label=grape,
            color=colors[grape],
            s=100,
            alpha=0.75,
            edgecolors="white",
            linewidth=1.4,
        )

    ax.set_title("Espaço de Separação: Polifenóis vs Aromas", fontsize=14, fontweight="bold")
    ax.set_xlabel("Energia da Banda de Polifenóis (Taninos)")
    ax.set_ylabel("Energia da Banda de Ésteres (Aromas)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    return fig, ax

plot_feature_space(df_resultados_sg)
plt.show()

## 10. Radar da identidade espectral média

O radar resume as cinco áreas médias em uma assinatura multidimensional para cada variedade.

In [ ]:
def plot_radar(df_medias, output_name="identidade_espectral.png"):
    labels = ["Açúcares", "Ácidos", "Polifenóis", "Proteínas", "Aromas"]
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
    colors = {"Cabernet": "#800020", "Shiraz": "#4B0082"}

    for grape in ["Cabernet", "Shiraz"]:
        values = df_medias.loc[grape].tolist()
        values += values[:1]
        ax.plot(angles, values, color=colors[grape], linewidth=2, label=grape)
        ax.fill(angles, values, color=colors[grape], alpha=0.22)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11, fontweight="bold")
    ax.set_title("Identidade Espectral Multidimensional", y=1.08, fontsize=15, fontweight="bold")
    ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1.1))
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / output_name, bbox_inches="tight")
    return fig, ax

plot_radar(df_sg_media)
plt.show()

## 11. Resumo dos resultados

Esta célula imprime alguns valores resumidos que podem ser usados no relatório ou na apresentação.

In [ ]:
summary = pd.DataFrame({
    "Cabernet SG": df_sg_media.loc["Cabernet"],
    "Shiraz SG": df_sg_media.loc["Shiraz"],
    "Cabernet MA": df_ma_media.loc["Cabernet"],
    "Shiraz MA": df_ma_media.loc["Shiraz"],
})
summary["Diferença SG Shiraz vs Cabernet (%)"] = (
    (summary["Shiraz SG"] - summary["Cabernet SG"]) / summary["Cabernet SG"] * 100
)

summary.index = AREA_LABELS
summary.round(3)

## 12. Arquivos gerados

Ao executar todas as células, os principais arquivos são atualizados em `results/`:

- `dados_iniciais_wine_01_rep1.png`, `dados_iniciais_wine_01_rep2.png`, `dados_iniciais_wine_01_rep3.png`
- `wine_01_media_triplicata.png`, `wine_15_syr_media_triplicatas.png`, `wine_30_cab_media_triplicatas.png`
- `wine_01.png`, `wine_15_syr.png`, `wine_30_cab.png`
- `passa_bandas_degraus_unitarios.png`
- `compara_passa_bandas_wine01_wine15.png`
- `concentracao_compostos_sg.png`, `concentracao_compostos_ma.png`
- `espaco_separacao_sg.png`
- `identidade_espectral.png`
- `savitzky_golay_corrigido.csv`, `media_movel_corrigido.csv`